In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings(action="ignore")

![SwiGLU](../picture/SwiGLU.jpg)

In [4]:
class BasicExpert(nn.Module):
    def __init__(self,feature_in,feature_out,hidden_dim):
        super().__init__()
        self.w_gate = nn.Linear(feature_in,hidden_dim,bias=False)
        self.w_up = nn.Linear(feature_in, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, feature_out, bias=False)
    
    def forward(self,x):
        gate = F.silu(self.w_gate(x)) # silu就是swish激活
        up = self.w_up(x)
        return self.w_down(gate*up)

![BasicMOE](../picture/BasicMOE.jpg)

In [15]:
class BasicMOE(nn.Module):
    def __init__(self, feature_in, feature_out,hidden_dim, expert_number):
        super().__init__()
        self.experts = nn.ModuleList(
            [
                BasicExpert(feature_in,feature_out,hidden_dim) for _ in range(expert_number)
            ]
        )

        self.gate = nn.Linear(feature_in,expert_number)

    def forward(self,x): 
        print("input x.shape:", x.shape)  # [batch, feature_in]
        # x - [batch,feature_in]
        # 1.每个样本给专家打分
        expert_weight = F.softmax(self.gate(x),dim=-1) # [batch,expert_number]
        print("expert_weight.shape:", expert_weight.shape)  # [batch, expert_number]

        # [batch,1,feature_out] - 后面要把多个expert的输出进行拼接
        expert_out_list = [
            expert(x).unsqueeze(1) for expert in self.experts 
        ]
        print("expert_out_list[0].shape:", expert_out_list[0].shape)  # [batch,1,feature_out]

        # 在维度1拼接起来，整合输出(batch,expert_number,feature_out)
        expert_output = torch.cat(expert_out_list,dim=1)
        print("expert_output.shape:", expert_output.shape)  # [batch,expert_number,feature_out]


        #2.expert输出做权重加和
        # [batch,1,expert_number]
        expert_weight = expert_weight.unsqueeze(1)
        print("expert_weight after unsqueeze.shape:", expert_weight.shape)  # [batch,1,expert_number]


        # [batch,1,feature_out]
        output = expert_weight @ expert_output
        print("output before squeeze.shape:", output.shape)  # [batch,1,feature_out]

        # 最后的输出将维度1去掉  - [batch,feature_out]
        return output.squeeze(1)

def test_basic_moe():
    torch.manual_seed(42)

    x = torch.rand(2, 4)   # batch=2, feature_in=4

    basic_moe = BasicMOE(
        feature_in=4,
        feature_out=3,
        hidden_dim=8,
        expert_number=2
    )

    out = basic_moe(x)

    # print("x.shape =", x.shape)
    # print("x =\n", x)
    # print("out.shape =", out.shape)
    print("out =\n", out)


test_basic_moe()


input x.shape: torch.Size([2, 4])
expert_weight.shape: torch.Size([2, 2])
expert_out_list[0].shape: torch.Size([2, 1, 3])
expert_output.shape: torch.Size([2, 2, 3])
expert_weight after unsqueeze.shape: torch.Size([2, 1, 2])
output before squeeze.shape: torch.Size([2, 1, 3])
out =
 tensor([[ 0.0014, -0.0681,  0.0398],
        [-0.0101, -0.0375,  0.0283]], grad_fn=<SqueezeBackward1>)


![SparseMOE](../picture/SparseMOE.jpg)

In [16]:
class MOERouter(nn.Module):
    def __init__(self, hidden_dim, expert_number, top_k):
        super().__init__()
        self.gate = nn.Linear(hidden_dim,expert_number)
        self.expert_number = expert_number
        self.top_k = top_k
        self.hidden_dim = hidden_dim

    def forward(self,hidden_states):
        #hidden_states.shape = (b * s, hidden_dim)
        # b*s个token，每个token有hidden_dim维的向量

        # 1.每个token对所有expert打分
        router_logits = self.gate(hidden_states) #[b*s,expert_number]

        # 2.变成概率
        # 用float精度更高
        router_probs = F.softmax(router_logits,dim=-1,dtype=torch.float) # [b*s,expert_number]
        
        # 3.选top_k个专家
        # 会返回最大的专家的下标和专家的值
        # [b*s,top_k]
        router_weights,selected_experts = torch.topk(
            router_probs,self.top_k,dim=-1
        )

        # 4.给top_kexpert分配权重 - 归一化
        # [b*s,top_k]
        router_weights = router_weights / router_weights.sum(dim=-1,keepdim=True)
        router_weights = router_weights.to(hidden_states.dtype)

        # 5.生成专家掩码
        # [b*s,top_k,expert_number]
        expert_mask = F.one_hot(
            selected_experts,
            num_classes = self.expert_number
        )
        # [expert_number,top_k,b*s]
        expert_mask = expert_mask.permute(2,1,0) 

        '''
            router_logits:保存路由打分
            router_weights:每个token分配给expert的权重
            selected_experts:每个token选了哪些epert
            expert_mask:按照expert找到对应token,做dispatch
        '''
        return router_logits,router_weights,selected_experts,expert_mask


In [17]:
class MOEConfig:
    def __init__(
            self, 
            hidden_dim, 
            expert_number, 
            top_k, 
            shared_experts_number=2,
        ):
        self.hidden_dim = hidden_dim
        self.expert_number = expert_number
        self.top_k = top_k
        self.shared_experts_number = shared_experts_number

In [ ]:
class SpareseMOE(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.hidden_dim = config.hidden_dim
        self.expert_number = config.expert_number
        self.top_k = config.top_k

        self.experts = nn.ModuleList(
            [
                BasicExpert(self.hidden_dim,self.hidden_dim,self.hidden_dim) for _ in range(self.expert_number)
            ]
        )

        self.router = MOERouter(self.hidden_dim,self.expert_number,self.top_k)

    def forward(self,x):
        # 1.展平
        batch_size,seq_len,hidden_dim = x.size() #[b,s,hidden_dim]
        print(batch_size,seq_len,hidden_dim)
        hidden_states = x.view(-1,hidden_dim) #[b*s,hidden_dim]

        # 2.利用router
        # 其中 selected_experts_indices shape 是 (b * s, top_k)
        # 其中 expert_mask shape 是 (expert_number, top_k, b * s)
        router_logits, router_weights, selected_experts_indices, expert_mask = self.router(hidden_states)
        

        final_hidden_states = torch.zeros(
            (batch_size * seq_len, hidden_dim),
            dtype=hidden_states.dtype,
            device=hidden_states.device
        ) # 存放结果 - [b*s,hidden_dim]

        # 遍历expert
        for idx in range(self.expert_number):
            # 获取对应专家
            expert_ontime = self.experts[idx]
            # 哪些token选中了当前expert
            index,top_x = torch.where(expert_mask[idx])
            #top_x：哪些 token 选中了当前 expert
            #index：当前 expert 对这些 token 来说是 top1 还是 top2
            '''
                idx   = tensor([0, 1, 1])
                top_x = tensor([2, 0, 5])
            '''
            #从 hidden_states 里，把 top_x 对应的那些 token 向量取出来
            current_state = hidden_states[top_x]
            # (selected_token_number, hidden_dim)

            #计算加权贡献
            current_hidden_state = expert_ontime(
                current_state
            ) # 做前向计算 - 当前expert对token的处理

            # router_weights - token分给当前expert的权重:[selected_token_number,] * [selected_token_number,hidden_dim](广播乘法)
            current_output = current_hidden_state * router_weights[top_x,index].unsqueeze(-1) # [selected_token_number,hidden_dim]
            # 此时会发生矩阵乘法

            # 加入到final_hidden_state
            # 按照 top_x 给出的 token 编号，把 current_hidden_states 的每一行，加回 final_hidden_states 的对应行
            final_hidden_states.index_add_(
                0, top_x, current_output.to(hidden_states.dtype))
            # 最后的token为多个expert的贡献和

        final_hidden_states = final_hidden_states.reshape(batch_size, seq_len, hidden_dim)

        # 每个 token 经过 top-k expert 加权融合后的新 hidden representation
        # 每个 token 对每个 expert 的原始打分
        return final_hidden_states,router_logits



def test_token_level_moe():
    x = torch.rand(2, 4, 16)
    config = MOEConfig(16, 2, 2)
    token_level_moe = SpareseMOE(config)
    out = token_level_moe(x)
    print(out[0].shape, out[1].shape)

test_token_level_moe()





2 4 16
torch.Size([2, 4, 16]) torch.Size([8, 2])


![SharedMOE](../picture/SharedMOE.jpg)

In [26]:
class ShareExpertMOE(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.moe_model = SpareseMOE(config)
        self.shared_experts = nn.ModuleList(
            [
                BasicExpert(
                    config.hidden_dim, config.hidden_dim,config.hidden_dim) for _ in range(config.shared_experts_number)
            ]
        )
    def forward(self,x):
        sparse_moe_out,router_logits = self.moe_model(x)

        shared_experts_out = [
            expert(x) for expert in self.shared_experts
        ] 

        # (shared_experts_number, b, s, hidden_dim)
        # 求和：(b, s, hidden_dim)
        shared_experts_out = torch.stack(
            shared_experts_out,dim=0
        ).sum(dim=0)

        return sparse_moe_out + shared_experts_out,router_logits


def test_share_expert_moe():
    x = torch.rand(2, 4, 16)
    config = MOEConfig(16, 2, 2)
    share_expert_moe = ShareExpertMOE(config)
    out = share_expert_moe(x)
    print(out[0].shape, out[1].shape)


test_share_expert_moe()

2 4 16
torch.Size([2, 4, 16]) torch.Size([8, 2])
